# Exercise 10.2: Z → ℓℓ analysis with a Crystal Ball fit revised

From *Programming in High Energy Particle Physics*, Chapter 10

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch10/ex10_2_solution.ipynb)

Exercise 10.2 Z → ℓℓ analysis with a Crystal Ball fit revised Implement a complete \(Z\to\ell^+\ell^-\) analysis in Python using uproot, awkward-array, vector, and hist. Apply selection criteria, compute the dilepton mass, and fit the Z peak with a Crystal Ball function using iminuit.

<details><summary>Hint</summary>

Start from Listing 10.5 . For the fit, use iminuit.cost.UnbinnedNLL(data, pdf) on the selected masses, or iminuit.cost.ExtendedBinnedNLL(counts, edges, cdf) with the histogram contents from h.values() and h.axes[0].edges . The Crystal Ball shape is available as scipy.stats.crystalball (with loc and scale , and its tail on the low side, which is what final-state radiation produces) or, faster, from the numba_stats package ( pip install numba-stats , module numba_stats.crystalball ). Restrict the fit to about 70–110 GeV and add an exponential background term if needed. Compare the fitted peak position with \(m_Z = 91.1876\) GeV.

</details>

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q uproot awkward vector hist iminuit scipy matplotlib numpy")

**Data:** seeded toy Z→ee events are written to a ROOT file and read back with uproot. The fit is a code demonstration, not a measurement.

### Step 1: Create or load a dilepton ROOT sample

In [ ]:
import numpy as np, awkward as ak, uproot
from scipy.stats import crystalball
rng=np.random.default_rng(42); N=3500; MZ=91.1876
target=crystalball.rvs(2.2,3.5,loc=MZ,scale=1.7,size=N,random_state=rng)
target=target[(target>70)&(target<110)]; pt=target/2
with uproot.recreate('toy_z.root') as f:
    f['Events']={'electron_pt':ak.Array([[float(v),float(v)] for v in pt]),
                 'electron_eta':ak.Array([[0.,0.] for v in pt]),
                 'electron_phi':ak.Array([[0.,float(np.pi)] for v in pt]),
                 'electron_charge':ak.Array([[1,-1] for v in pt])}
events=uproot.open('toy_z.root')['Events'].arrays(library='ak')

### Step 2: Apply an opposite-sign selection and histogram

In [ ]:
import vector, hist
vector.register_awkward()
ele=ak.zip({'pt':events.electron_pt,'eta':events.electron_eta,'phi':events.electron_phi,
            'mass':ak.ones_like(events.electron_pt)*0.000511,'charge':events.electron_charge},with_name='Momentum4D')
selected=ele[(ele.pt>20)&(abs(ele.eta)<2.5)]
pairs=ak.combinations(selected,2,fields=['l1','l2'])
os=pairs[(pairs.l1.charge*pairs.l2.charge)<0]
masses=ak.to_numpy(ak.flatten((os.l1+os.l2).mass)); masses=masses[(masses>70)&(masses<110)]
h=hist.Hist(hist.axis.Regular(40,70,110,name='mass',label='Dilepton mass [GeV]')); h.fill(mass=masses)
assert len(masses)>1000

### Step 3: Fit a low-tail Crystal Ball shape

In [ ]:
from iminuit import Minuit
from iminuit.cost import UnbinnedNLL
import matplotlib.pyplot as plt
def pdf(x,beta,n,loc,scale):
    z=crystalball.cdf(110,beta,n,loc=loc,scale=scale)-crystalball.cdf(70,beta,n,loc=loc,scale=scale)
    return crystalball.pdf(x,beta,n,loc=loc,scale=scale)/z
fit=Minuit(UnbinnedNLL(masses,pdf),beta=2,n=3,loc=91,scale=2)
fit.limits['beta']=(.5,10); fit.limits['n']=(1.1,20); fit.limits['loc']=(85,98); fit.limits['scale']=(.3,6)
fit.migrad(); fit.hesse(); assert fit.valid
print('Fitted location [GeV]:',round(fit.values['loc'],3),'+/-',round(fit.errors['loc'],3),'PDG:',MZ)
fig,ax=plt.subplots(figsize=(8,4)); h.plot1d(ax=ax,label='toy Z→ee')
x=np.linspace(70,110,300); ax.plot(x,pdf(x,*fit.values)*len(masses),label='Crystal Ball fit')
ax.axvline(MZ,ls='--',color='black',label='PDG'); ax.set(xlabel='Dilepton mass [GeV]',ylabel='Toy events / GeV'); ax.legend(); plt.show()

The fitted Crystal Ball location should be compatible with the injected 91.1876 GeV. The toy low-mass tail mimics final-state radiation; detector and calibration systematics are absent.